In [ ]:
# Required for debug only. If you have data_fast_insights installed, you can delete this cell.
import sys
from pathlib import Path
import os

sys.path.append(str(Path(os.getcwd()).parent.parent))
sys.path.append(str(Path(os.getcwd()).parent))

In [ ]:
%config InlineBackend.figure_format = 'svg'

Getting the data

In [ ]:
import pandas as pd
from sklearn import datasets

from data_fast_insights import BinaryDependenceModelData
import data_fast_insights.calculations as calc

from data_fast_insights.plotting import plot_segments_basic_info, create_feature_segments_plot_plotly

In [ ]:
raw_data = datasets.fetch_california_housing()
# print(raw_data['DESCR'])

In [ ]:
df = pd.DataFrame(raw_data['data'], columns=raw_data['feature_names'])
df['MedianHouseValue'] = raw_data['target']
df.head()

Let's add a categorical column for the display of how module works with these

In [ ]:
df['MedInc_gt_3'] = df['MedInc'] > 3.0

## Using Data Fast Insights

Initializing model data

In [ ]:
dmd = BinaryDependenceModelData(
    base_data=df,
    cat_cols={'MedInc_gt_3'},
    num_cols={'MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude'},
    y_name='MedianHouseValue',
    y_quantile=0.5,
    # inverse_goal=True
)

Getting bins for numeric variables, optimizing for Information Value. 

In [ ]:
num_bins = calc.make_bins(model_data=dmd)

Converting variables

In [ ]:
dmd.convert_to_binary(bins=num_bins)

Calculating group importance and other metrics

In [ ]:
res = calc.calculate_dependence(model_data=dmd,
                                # sort_from_best_to_worst=False
                                )

Getting data about segments (dataframe is sorted by importance)

In [ ]:
res[res['perc_of_total'] > 5].head().round(1)

### Plotting basic info about features segments

With increasing the occupancy number house value drops (see AveOccup feature)

There is a significant drop of house value in blocks 
located at Longitude from -121 up to -119 (not including -119), which requires further research. (see Longtitude feature)

Blocks with residents having highest income contain the most highly valued houses (see MedInc feature)

In [ ]:
create_feature_segments_plot_plotly(
    info_df=res,
    segment_order=dmd.col_links.keys(),
    target_name='Median House Value',
)

### Combinations of segments

In [ ]:
dmd.remove_segment_combinations()

Please note that since combinations are added to an existing objects,
calling add_combinations_from_decision_tree() multiple times with different parameters
might create new segments, but will not erase ones previously created with this method.

In [ ]:
dmd.add_combinations_from_decision_tree(comb_max_size=None, comb_max_count=15)

add_combinations_from_existing_segments() also adds segments to existing ones, but doesn't erase previously created.
Segments of the same depth will be the same and thus just rewritten, but for example if you call this with comb_max_size=3, then again with comb_max_size=2,
these with combination size of 3 will stay.

In [ ]:
dmd.add_combinations_from_existing_segments(2)

In [ ]:
res = calc.calculate_dependence(model_data=dmd)

In [ ]:
# import line_profiler
# line_profiler.profile.enable()
# calc.calculate_dependence(model_data=dmd)
# line_profiler.profile.show()
# line_profiler.profile.disable()
# # print(open("profile_output.txt").read())

In [ ]:
res.head()

In [ ]:
res[res['base_col']=='<from_decision_tree_custom>'].head()

In [ ]:
create_feature_segments_plot_plotly(
    info_df=res[res.index.str.contains('_AND_')],  # only show combined features
    # info_df=res,
    segment_order=dmd.col_links.keys(),
    target_name='Median House Value',
)